## Embed Name + Genres + Description $\rightarrow$ Genres, Tags to Chromadb

**Warning**: if you really wants to run this and wait for about **150 minutes** then you should run this file. Otherwise, you should have a docker ChromaDB that contains all data

### Import Necessary Lib

In [29]:
import pandas as pd
from utils.preprocess.imputation.genres_tags.schemas import InsertRequest, RetrieveRequest
from utils.preprocess.imputation.genres_tags.func import insert_tag_batch, retrieve_tag_list

### Load Dataset

In [30]:
DATA_PATH = 'data/steam-games/SteamGames_Clean_1.csv'
df = pd.read_csv(DATA_PATH)

### Extract Necessary Features Only

In [31]:
df = df[["Name", "Type", "Genres", "Description", "Tags", "Rank"]]
df_no_dup = df.drop_duplicates()
# We will use these features to predict tags
df_imputated = df_no_dup.dropna(subset=["Name", "Type", "Genres", 'Description'])

### Split to training, predict

In [32]:
df_training= df_imputated.dropna(subset=["Tags"])
df_predict = df_imputated[df["Tags"].isna()]

df_training = df_training.reset_index(drop=True)
df_predict = df_predict.reset_index(drop=True)

/tmp/ipykernel_163876/442432523.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_predict = df_imputated[df["Tags"].isna()]


### Put Genres and Tags to ChromaDB
**You should not** run the cell below because it will take a long time to do it. You should have a prerun, already inserted version of **chromadb** storage( I will send later)

In [33]:
# Only run this once to put tags into ChromaDB, no need for second run
# But if you are break enough just run it anyway:)))
items = []
batch_size = 1000
batch_index = 0        # continue from this batch

for index, row in df_training.iterrows():
    if index < batch_index * batch_size:
        continue
    
    item = InsertRequest(
        name=row['Name'],
        genres=row['Genres'].split(','),
        description=row['Description'],
        rank=index + 1,
        tag_list=row['Tags'].split(',')
    )

    items.append(item)

    if len(items) % batch_size == 0 and len(items) > 0:
        print(f"Insert batch {index // 1000}")
        insert_tag_batch(items)
        print("Insert complete")
        items.clear()

    

### Try to Retrive

In [7]:
retrieve = RetrieveRequest(
    name=df_predict['Name'][6],
    genres=df_predict['Genres'][6].split(','),
    description=df_predict['Description'][6],
    top_k= 5,
    top_tag=10
)
response = retrieve_tag_list(retrieve)

print(retrieve)
print(response)

name='Grand Theft Auto V Enhanced' genres=['Action', 'Adventure', 'Racing'] description="Welcome to Los Santos When a young street hustler, a retired bank robber, and a terrifying psychopath find themselves entangled with some of the most frightening and deranged elements of the criminal underworld, the U.S. government, and the entertainment industry, they must pull off a series of dangerous heists to survive in a ruthless city in which they can trust nobody — least of all each other. Current PC players can transfer both GTAV Story Mode progress and GTA Online characters and progression with a one-time migration from the legacy version of GTAV to GTAV Enhanced. Stunning Visuals Enhanced levels of fidelity and performance with new graphics modes including ray tracing features such as ambient occlusion and global illumination, ray traced shadows and reflections, support for AMD FSR and NVIDIA DLSS, and more.* Faster Loading Quicker access to the action as the world of Los Santos and Blai

In [34]:
TEST_DATA_PATH = "data/steam-games/SteamGames_test_100.csv"
df_test = pd.read_csv(TEST_DATA_PATH)
df_test = df_test[["Name", "Type", "Genres", "Description", "Tags", "Rank"]]
df_test.drop_duplicates(inplace=True)
df_test.dropna(inplace=True)

In [36]:
# Run test
predicts = []

for _, row in df_test.iterrows():
    request = RetrieveRequest(
        name=row['Name'],
        genres=row['Genres'].split(','),
        description=row['Description'],
        top_k=10,
        top_tag=10
    )
    predict = retrieve_tag_list(request)
    predicts.append(predict.tag_list)

### Evaluate Retrive Accuracy

In [42]:
# predicts = [predict.tag_list for predict in predicts]
predicts
df_test['Tags']
y_true = df_test['Tags'].str.split(',')
y_true

1     [FPS, Shooter, Multiplayer, Competitive, Actio...
2     [Extraction Shooter, Multiplayer, PvP, PvE, Th...
4     [Free to Play, Multiplayer, Battle Royale, FPS...
5     [Turn-Based Tactics, Roguelite, Turn-Based Str...
7     [Free to Play, Multiplayer, Hero Shooter, Thir...
                            ...                        
95    [Horror, Multiplayer, Co-op, Survival Horror, ...
96    [Open World, Survival, Creature Collector, Mul...
97    [Tactical, Realistic, FPS, Shooter, Multiplaye...
98    [Relaxing, Cute, Open World, Casual, RPG, Co-o...
99    [Strategy, World War II, Grand Strategy, War, ...
Name: Tags, Length: 73, dtype: object

### Label every missing missing value in 'Tags'

In [ ]:
def evaluate_tag_prediction(y_true_single, y_pred_single):
    # Ensure inputs are sets of clean string tags
    set_true = set(y_true_single) if isinstance(y_true_single, (list, set)) else set()
    set_pred = set(y_pred_single) if isinstance(y_pred_single, (list, set)) else set()

    intersection = len(set_true & set_pred)
    union = len(set_true | set_pred)

    jaccard = intersection / union if union > 0 else 0.0
    precision = intersection / len(set_pred) if len(set_pred) > 0 else 0.0
    recall = intersection / len(set_true) if len(set_true) > 0 else 0.0

    return {"jaccard": jaccard, "precision": precision, "recall": recall}

# 1. Split Tags string into lists of clean tags (stripping whitespace)
y_true = df_test['Tags'].dropna().apply(lambda x: [tag.strip() for tag in str(x).split(',') if tag.strip()])

# 2. Iterate row-by-row across y_true and predicts
metrics = [
    evaluate_tag_prediction(true_tags, pred_tags)
    for true_tags, pred_tags in zip(y_true, predicts)
]

# 3. Aggregate into a clean summary DataFrame
metrics_df = pd.DataFrame(metrics)

print("Mean Jaccard:   ", round(metrics_df['jaccard'].mean(), 4))
print("Mean Precision: ", round(metrics_df['precision'].mean(), 4))
print("Mean Recall:    ", round(metrics_df['recall'].mean(), 4))

Mean Jaccard:    0.3931
Mean Precision:  0.8055
Mean Recall:     0.4637


In [28]:
y_true

1     [FPS, Shooter, Multiplayer, Competitive, Actio...
2     [Extraction Shooter, Multiplayer, PvP, PvE, Th...
4     [Free to Play, Multiplayer, Battle Royale, FPS...
5     [Turn-Based Tactics, Roguelite, Turn-Based Str...
7     [Free to Play, Multiplayer, Hero Shooter, Thir...
                            ...                        
95    [Horror, Multiplayer, Co-op, Survival Horror, ...
96    [Open World, Survival, Creature Collector, Mul...
97    [Tactical, Realistic, FPS, Shooter, Multiplaye...
98    [Relaxing, Cute, Open World, Casual, RPG, Co-o...
99    [Strategy, World War II, Grand Strategy, War, ...
Name: Tags, Length: 73, dtype: object